In [ ]:
from PIL import Image, ImageDraw, ImageFont, ImageSequence

In [ ]:
in_path = "Animation.gif"
out_path = "Animation_text_small.gif"

font_size = 60 
scale = 0.75

im = Image.open(in_path)

# Determine new size from original size
w, h = im.size
new_size = (max(1, int(w * scale)), max(1, int(h * scale)))

frames = []
durations = []
disposal = im.info.get("disposal", 2)  # 2 is common (clear previous frame before drawing)

for frame in ImageSequence.Iterator(im):
    # Convert to RGBA for drawing
    fr = frame.convert("RGBA")
    draw = ImageDraw.Draw(fr)

    lpos = [(20, 20)          # Coordinates from top-left
    ,(550, 20)          # Coordinates from top-left
    ,(1180, 20)          # Coordinates from top-left
    ]

    ltext = ["original", "compressed", "compressed+cog"]

    for pos, text in zip(lpos, ltext):
        x, y = pos
        fill = "black"
        outline = "white"

        # Draw outline (for better readability)
        for dx in (-2, 0, 2):
            for dy in (-2, 0, 2):
                if dx == 0 and dy == 0: continue
                draw.text((x+dx, y+dy), text, fill=outline, font=ImageFont.load_default(size=font_size))

        draw.text((x, y), text, fill=fill, font=ImageFont.load_default(size=font_size))

    # Resize (high quality)
    fr_small = fr.resize(new_size, resample=Image.Resampling.LANCZOS)
    # Convert frame back to palette mode (GIF is basically palette-based)
    # fr_p = fr.convert("P", palette=Image.Palette.ADAPTIVE)
    fr_p = fr_small.convert("P", palette=Image.Palette.ADAPTIVE)

    frames.append(fr_p)
    # Set frame duration to 1/3 of original
    durations.append(frame.info.get("duration", im.info.get("duration", 100))/3)  

# Loop info
loop = im.info.get("loop", 0)

frames[0].save(
    out_path,
    save_all=True,
    append_images=frames[1:],
    duration=durations,
    loop=loop,
    disposal=disposal,
    optimize=True
)

print("Saved:", out_path)